# Orders Auto Loader - Bronze Layer

Incrementally ingests orders data from Volume source into the bronze layer using Auto Loader.

In [0]:
# Source and target configuration
source_path = "/Volumes/retailnova/bronze/retailnova_source/retailnova_datasets/orders/"
target_table = "retailnova.bronze.orders"
checkpoint_path = "/Volumes/retailnova/bronze/retailnova_source/retailnova_datasets/_checkpoints/orders_bronze"

# readStream
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")  
  .option("cloudFiles.schemaLocation", checkpoint_path + "/_schema")
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
  .load(source_path)
)

# writeStream
(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .option("mergeSchema", "true")
  .trigger(availableNow=True) 
  .toTable(target_table)
)

In [0]:
%sql
SELECT COUNT(*) FROM retailnova.bronze.orders

COUNT(*)
1003000


In [0]:
# Preview the ingested orders data
df_orders = spark.table("retailnova.bronze.orders")
print(f"Total rows: {df_orders.count():,}")
print(f"\nSchema:")
df_orders.printSchema()
print(f"\nSample data:")
display(df_orders.limit(10))

Total rows: 1,003,000

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- quantity: double (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)


Sample data:


order_id,customer_id,product_id,store_id,order_timestamp,quantity,unit_price,discount_amount,status,payment_method,updated_at,_rescued_data
O0100301,C038298,P008400,ST0458,2026-02-27T18:48:17.000Z,1.0,100.0,22.75,Completed,Net Banking,2026-03-02T08:48:17.000Z,null
O0100302,C080604,P006018,ST0040,2026-01-13T14:08:08.000Z,4.0,208.01,170.69,Pending,Debit Card,2026-01-16T07:08:08.000Z,null
O0100303,C038063,P009021,ST0353,2026-03-07T02:01:53.000Z,1.0,3573.91,387.61,Pending,Cash,2026-03-07T07:01:53.000Z,null
O0100304,C069218,P000992,ST0453,2026-03-09T03:17:08.000Z,3.0,4140.68,1960.54,Completed,Credit Card,2026-03-09T22:17:08.000Z,null
O0100305,C033646,P003467,ST0268,2026-07-09T16:01:57.000Z,5.0,486.77,485.4,Completed,Debit Card,2026-07-12T11:01:57.000Z,null
O0100306,C059932,P005021,ST0038,2026-02-08T20:57:27.000Z,1.0,6533.41,1426.32,Completed,Credit Card,2026-02-09T14:57:27.000Z,null
O0100307,C069760,P008270,ST0130,2026-08-10T23:20:30.000Z,5.0,762.23,667.87,Shipped,Credit Card,2026-08-12T11:20:30.000Z,null
O0100308,C055527,P003918,ST0031,2026-06-06T09:23:01.000Z,3.0,1648.93,142.76,Completed,UPI,2026-06-08T00:23:01.000Z,null
O0100309,C084886,P009036,ST0256,2026-07-22T11:51:30.000Z,2.0,928.69,320.72,Completed,UPI,2026-07-25T07:51:30.000Z,null
O0100310,C085593,P007535,ST0113,2026-03-08T05:58:21.000Z,1.0,1339.67,218.04,Completed,Cash,2026-03-10T16:58:21.000Z,null
